# Erdős-Straus Covering Lemma Prover
## Kaggle Kernel — Omega Solver + Decision Tree + Period Analysis

**Status:** Empirical verification complete up to 5×10⁶ (19,224 exceptional primes, 0 failures, A ≤ 159).

**Goal:** Prove that minimal working A for exceptional Tier 3 primes depends only on p modulo a fixed finite modulus M.

In [ ]:
# SETUP
import json, time, math, os, sys
from datetime import datetime
from pathlib import Path
from collections import Counter
from itertools import combinations

try:
    import subprocess
    r = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                       capture_output=True, text=True, timeout=5)
    GPU = r.stdout.strip() if r.returncode==0 else 'CPU'
except: GPU = 'CPU'

print(f"GPU: {GPU}")
print(f"Start: {datetime.now().isoformat()}")
print(f"Python: {sys.version}")

## 1. Core Functions

In [ ]:
# CANDIDATE A VALUES — the 22-element immune set
CANDIDATE_A = [7, 11, 15, 19, 23, 31, 39, 43, 47, 51, 59, 67, 71, 79, 83, 87, 95, 103, 107, 111, 127, 159]

def factorize(n):
    """Factorize n into {prime: exponent} dict."""
    m = n; res = {}; q = 2
    while q * q <= m:
        while m % q == 0:
            res[q] = res.get(q, 0) + 1; m //= q
        q += 1 if q == 2 else 2
    if m > 1:
        res[m] = res.get(m, 0) + 1
    return res

def divisors_from_factors(factors):
    """Generate all divisors from factor dict."""
    divs = [1]
    for prime, exp in factors.items():
        cur = []; p_pow = 1
        for _ in range(exp + 1):
            for d in divs: cur.append(d * p_pow)
            p_pow *= prime
        divs = cur
    return divs

def is_exceptional(p):
    """Check if p is Tier 3 exceptional: p ≡ 1 (mod 12), c=(p+3)/4 has no prime factor ≡ 2 (mod 3)."""
    if p % 12 != 1: return False
    c = (p + 3) // 4; m = c; q = 2
    while q * q <= m:
        if m % q == 0:
            if q % 3 == 2: return False
            while m % q == 0: m //= q
        q += 1 if q == 2 else 2
    if m > 1 and m % 3 == 2: return False
    return True

def check_A(p, A):
    """Omega solver: check if shift A works for prime p."""
    n = p * p
    if (n + A) % 4 != 0: return False, None
    x = (n + A) // 4; nx = n * x; target_mod = (-nx) % A
    fac = factorize(x)
    for q in list(fac): fac[q] *= 2
    fac[p] = fac.get(p, 0) + 4
    for d in divisors_from_factors(fac):
        if d % A == target_mod:
            y = (nx + d) // A
            z = (nx + nx * nx // d) // A
            if y > 0 and z > 0 and 4 * x * y * z == n * (x*y + x*z + y*z):
                return True, d
    return False, None

def find_min_A(p, max_m=200):
    """Find minimal working A for prime p."""
    for m in range(max_m):
        A = 4 * m + 3
        ok, d = check_A(p, A)
        if ok: return A
    return None

print(f"Core functions ready. Candidate A set: {CANDIDATE_A}")
print(f"Max A = {max(CANDIDATE_A)}, {len(CANDIDATE_A)} candidates")

## 2. Sieve Exceptional Primes

In [ ]:
def sieve_exceptional(limit):
    """Find all exceptional primes up to limit using sieve of Eratosthenes."""
    is_prime = bytearray(b'\x01') * (limit + 1)
    is_prime[0:2] = b'\x00\x00'
    for i in range(2, int(limit**0.5) + 1):
        if is_prime[i]:
            step = i; start = i * i
            is_prime[start:limit+1:step] = b'\x00' * ((limit - start) // step + 1)
    exceptional = []
    for p in range(13, limit + 1):
        if is_prime[p] and is_exceptional(p):
            exceptional.append(p)
    return exceptional

# Test with small limit
test_exc = sieve_exceptional(1000)
print(f"Exceptional primes up to 1000: {len(test_exc)}")
print(f"First 20: {test_exc[:20]}")

## 3. Compute A_min Distribution

In [ ]:
# CONFIG
LIMIT = 5_000_000  # Sieve limit
OUTPUT = Path("/kaggle/working/covering_lemma_results.json")

print(f"Sieving exceptional primes up to {LIMIT:,}...")
t0 = time.perf_counter()
exceptional = sieve_exceptional(LIMIT)
t1 = time.perf_counter()
print(f"Found {len(exceptional)} exceptional primes in {t1-t0:.1f}s")
print()

print("Computing A_min for all exceptional primes...")
cache = {}
t0 = time.perf_counter()
prime_data = []  # (p, A_min)
failures = []

for idx, p in enumerate(exceptional):
    if p in cache:
        A = cache[p]
    else:
        A = find_min_A(p, max_m=200)
        cache[p] = A
    if A is not None:
        prime_data.append((p, A))
    else:
        failures.append(p)
    if (idx + 1) % 5000 == 0:
        t = time.perf_counter() - t0
        rate = (idx + 1) / max(t, 0.01)
        print(f"  [{idx+1}/{len(exceptional)}] {t:.1f}s elapsed, {rate:.0f} primes/s")

t1 = time.perf_counter()
print(f"\nComputed A_min for {len(prime_data)} primes in {t1-t0:.1f}s")
print(f"Failures: {len(failures)}")
if failures:
    print(f"  Failed primes: {failures[:20]}")

In [ ]:
# DISTRIBUTION
A_dist = Counter(A for _, A in prime_data)
print("A_min distribution:")
print(f"{'A':>4} {'m':>4} {'Count':>8} {'%':>7}")
print("-" * 30)
for A in sorted(A_dist):
    m = (A - 3) // 4
    pct = 100 * A_dist[A] / len(prime_data)
    print(f"{A:>4} {m:>4} {A_dist[A]:>8} {pct:>6.1f}%")
print(f"{'Total':>8} {len(prime_data):>8}")
print()

# Verify all A in candidate set
all_in_set = all(A in CANDIDATE_A for _, A in prime_data)
print(f"All A_min in candidate set: {all_in_set}")
print(f"Max A_min: {max(A for _, A in prime_data)}")

## 4. Decision Tree Verification (Levels 1–3)

In [ ]:
# DECISION TREE — Levels 1-3
print("Decision tree verification:")
print()

# Level 1: Modulus 7
A7 = [p for p, A in prime_data if A == 7]
A7_mod7 = set(p % 7 for p in A7)
print(f"Level 1 (A=7): {len(A7)} primes ({100*len(A7)/len(prime_data):.1f}%)")
print(f"  p mod 7 ∈ {sorted(A7_mod7)}")
print(f"  Expected: {{3, 5, 6}}")
print(f"  Match: {A7_mod7 == {3, 5, 6}}")
print()

# Level 2: Modulus 11 (only for primes not resolved at Level 1)
A11 = [p for p, A in prime_data if A == 11]
A11_mod11 = set(p % 11 for p in A11)
print(f"Level 2 (A=11): {len(A11)} primes ({100*len(A11)/len(prime_data):.1f}%)")
print(f"  p mod 11 ∈ {sorted(A11_mod11)}")
print(f"  Expected: {{2, 6, 7, 10}}")
print(f"  Match: {A11_mod11 == {2, 6, 7, 10}}")
print()

# Level 3: Modulus 5
A15 = [p for p, A in prime_data if A == 15]
A15_mod5 = set(p % 5 for p in A15)
print(f"Level 3 (A=15): {len(A15)} primes ({100*len(A15)/len(prime_data):.1f}%)")
print(f"  p mod 5 ∈ {sorted(A15_mod5)}")
print(f"  Expected: {{3}}")
print(f"  Match: {A15_mod5 == {3}}")
print()

# Combined coverage
covered = len(A7) + len(A11) + len(A15)
print(f"Levels 1-3 coverage: {covered}/{len(prime_data)} ({100*covered/len(prime_data):.1f}%)")
print(f"Remaining for Levels 4+: {len(prime_data) - covered}")

## 5. Period Bounds

In [ ]:
def compute_period_bound_for_A(A):
    """Theoretical upper bound on the period of P_A(p)."""
    f = factorize(A)
    Ma = 4 * A  # base from congruence condition
    for q in f:
        Ma = Ma * q ** (f[q] + 1) // math.gcd(Ma, q ** (f[q] + 1))
    return Ma

print("Period bounds M_A for each candidate A:")
print(f"{'A':>4} {'M_A bound':>15} {'Factorization':>20}")
print("-" * 45)

periods = {}
for A in CANDIDATE_A:
    Ma = compute_period_bound_for_A(A)
    periods[A] = Ma
    fac = factorize(Ma)
    fac_str = " × ".join(f"{q}^{e}" for q, e in sorted(fac.items()))
    print(f"{A:>4} {Ma:>15,} {fac_str:>20}")

# Full period
M_full = 1
for A, Ma in periods.items():
    M_full = M_full * Ma // math.gcd(M_full, Ma)
print(f"\nFull period M = lcm(M_A) = {M_full:,}")
print(f"Factorization: {factorize(M_full)}")

## 6. Modulus Search

In [ ]:
def check_modulus(prime_data, M):
    """Check whether A_min(p) depends only on p mod M."""
    mapping = {}
    inconsistent = 0
    for p, A in prime_data:
        r = p % M
        if r in mapping:
            if mapping[r] != A:
                inconsistent += 1
        else:
            mapping[r] = A
    collisions = len(prime_data) - len(mapping)
    return inconsistent == 0, collisions, len(mapping), inconsistent

# Search over lcm combinations of decision tree moduli
DECISION_MODULI = [7, 11, 5, 13, 17, 37]

candidates = set()
for r in range(1, len(DECISION_MODULI) + 1):
    for combo in combinations(DECISION_MODULI, r):
        M = 1
        for q in combo:
            M = M * q // math.gcd(M, q)
        candidates.add(M)
    for combo in combinations(DECISION_MODULI, r):
        M = 12
        for q in combo:
            M = M * q // math.gcd(M, q)
        candidates.add(M)

print(f"Testing {len(candidates)} candidate moduli...")
print(f"{'M':>10} {'Unique':>8} {'Collisions':>10} {'Inconsistent':>12} {'Status':>8}")
print("-" * 55)

results = []
for M in sorted(candidates):
    consistent, collisions, unique, inc = check_modulus(prime_data, M)
    results.append((M, consistent, collisions, unique, inc))
    status = "PASS" if consistent and collisions >= 5 else ("CHECK" if consistent else "FAIL")
    print(f"{M:>10,} {unique:>8} {collisions:>10} {inc:>12} {status:>8}")

## 7. Formal Residue Mapping

In [ ]:
# Find best consistent modulus
best_results = [(M, c, u, i) for M, ok, c, u, i in results if ok]

if best_results:
    best = max(best_results, key=lambda x: x[1])  # most collisions
    M_best, best_coll, best_unique, best_inc = best
    print(f"Best modulus: M = {M_best:,}")
    print(f"  Unique residues: {best_unique}")
    print(f"  Collisions: {best_coll}")
    print(f"  Inconsistencies: {best_inc}")
    print()

    # Build full mapping
    mapping = {}
    for p, A in prime_data:
        r = p % M_best
        mapping[r] = A

    # Group by A
    A_residues = {}
    for r, A in mapping.items():
        A_residues.setdefault(A, []).append(r)

    print("Residue mapping:")
    for A in sorted(A_residues):
        residues = sorted(A_residues[A])
        print(f"  A={A:>3}: {len(residues)} residues")
        if len(residues) <= 30:
            for r in residues:
                sample_p = next(p for p, a in prime_data if p % M_best == r)
                print(f"         r={r:>6} (e.g. p={sample_p})")

    # Decision tree verification
    print()
    print("Decision tree verification:")
    A7_res = [r for r, a in mapping.items() if a == 7]
    A11_res = [r for r, a in mapping.items() if a == 11]
    A15_res = [r for r, a in mapping.items() if a == 15]
    print(f"  A=7:  residues mod 7 = {sorted(set(r % 7 for r in A7_res))}")
    print(f"  A=11: residues mod 11 = {sorted(set(r % 11 for r in A11_res))}")
    print(f"  A=15: residues mod 5 = {sorted(set(r % 5 for r in A15_res))}")
else:
    print("No consistent modulus found. Consider larger prime limit.")

## 8. Save Results

In [ ]:
# SAVE RESULTS
results_data = {
    "timestamp": datetime.now().isoformat(),
    "gpu": GPU,
    "limit": LIMIT,
    "num_exceptional": len(exceptional),
    "num_solved": len(prime_data),
    "num_failures": len(failures),
    "max_A": max(A for _, A in prime_data),
    "A_distribution": {str(A): count for A, count in sorted(A_dist.items())},
    "period_bounds": {str(A): Ma for A, Ma in periods.items()},
    "M_full": M_full,
    "best_modulus": M_best if best_results else None,
    "decision_tree": {
        "level1_A7_mod7": sorted(A7_mod7) if A7_mod7 else [],
        "level2_A11_mod11": sorted(A11_mod11) if A11_mod11 else [],
        "level3_A15_mod5": sorted(A15_mod5) if A15_mod5 else [],
        "levels123_coverage_pct": round(100*covered/len(prime_data), 1) if prime_data else 0
    }
}

OUTPUT.write_text(json.dumps(results_data, indent=2))
print(f"Results saved to {OUTPUT}")
print()

# Summary
print("=" * 60)
print("COVERING LEMMA PROVER — SUMMARY")
print("=" * 60)
print(f"Primes analyzed: {len(prime_data):,}")
print(f"Failures: {len(failures)}")
print(f"Max A_min: {max(A for _, A in prime_data)}")
print(f"Levels 1-3 coverage: {100*covered/len(prime_data):.1f}%")
if best_results:
    print(f"Best modulus: M = {M_best:,}")
print(f"Full period bound: M = {M_full:,}")
print("=" * 60)